<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [2]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [4]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
df['ctr'] = df['gsc_clicks']/df['gsc_impressions']
df['ctr'].fillna(0, inplace=True)

/tmp/ipykernel_32049/4243852079.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['ctr'].fillna(0, inplace=True)


In [6]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [7]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [8]:
df = df.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    ga4_pageviews=('ga4_pageviews', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_users=('ga4_users', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
    ctr=('ctr', 'mean')
)

In [9]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id','gsc_impressions_feb', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [10]:
conditions = [
    df['trend_pct'] < -50,
    (df['trend_pct'] >= -50) & (df['trend_pct'] < -15),
    (df['trend_pct'] >= -15) & (df['trend_pct'] <= 15),
    (df['trend_pct'] > 15) & (df['trend_pct'] <= 50),
    df['trend_pct'] > 50
]
ranks = ['5-Sharp decline', '4-Mild decline', '3-Flat', '2-Mild growth', '1-Strong growth']
df['trend_dir'] = np.select(conditions, ranks, default=None)

In [11]:
df['trend_dir'].unique()

array([None, '4-Mild decline', '1-Strong growth', '5-Sharp decline',
       '3-Flat', '2-Mild growth'], dtype=object)

In [12]:
display(df.groupby('trend_dir').agg(
    {
        'gsc_sum_position': ['mean', 'count'],
        'gsc_avg_position': ['mean', 'count'],
        'gsc_impressions' : ['mean', 'count'],
        'gsc_clicks' : ['mean', 'count'],
        'ga4_pageviews' : ['mean', 'count'],
        'ga4_sessions' : ['mean', 'count'],
        'ga4_users' : ['mean', 'count'],
        'ga4_engaged_sessions' : ['mean', 'count'],
        'ctr' : ['mean', 'count'],
    }
))

gsc_sum_position        gsc_avg_position         \
                            mean  count             mean  count   
trend_dir                                                         
1-Strong growth      7444.074363   9225        14.259258   9225   
2-Mild growth       13366.686321   9328        12.596824   9328   
3-Flat              20963.225323  18671        12.339732  18671   
4-Mild decline      34521.747223  31692        15.397712  31692   
5-Sharp decline     38236.897774  36204        20.126775  36204   

                gsc_impressions        gsc_clicks        ga4_pageviews         \
                           mean  count       mean  count          mean  count   
trend_dir                                                                       
1-Strong growth      863.653117   9225   2.249973   9225      7.688672   9225   
2-Mild growth       1839.991209   9328   5.681604   9328      9.975986   9328   
3-Flat              2803.647153  18671   9.738097  18671     13.318462  18671   
4-Mild decline      2803.778619  31692   7.001988  31692     14.453017  31692   
5-Sharp decline     2613.579825  36204   7.275439  36204     11.737349  36204   

                ga4_sessions         ga4_users        ga4_engaged_sessions  \
                        mean  count       mean  count                 mean   
trend_dir                                                                    
1-Strong growth     7.249214   9225   7.198591   9225             0.086179   
2-Mild growth        9.08319   9328   8.936428   9328             0.160806   
3-Flat             11.706765  18671  11.466499  18671             0.263135   
4-Mild decline     12.573489  31692  12.345071  31692             0.291051   
5-Sharp decline    10.220418  36204   9.898575  36204             0.253701   

                             ctr         
                 count      mean  count  
trend_dir                                
1-Strong growth   9225  0.002225   9225  
2-Mild growth     9328  0.002575   9328  
3-Flat           18671  0.002713  18671  
4-Mild decline   31692  0.002314  31692  
5-Sharp decline  36204  0.002422  36204

In [13]:
df.groupby('trend_dir')['gsc_impressions_feb'].describe()

,count,mean,std,min,25%,50%,75%,max
trend_dir,,,,,,,,
1-Strong growth,9225.0,863.653117,3181.702594,30.0,73.0,204.0,645.00,134984.0
2-Mild growth,9328.0,1839.991209,4700.314286,30.0,151.0,491.0,1651.25,142304.0
3-Flat,18671.0,2803.647153,6455.424658,30.0,222.0,795.0,2757.00,170808.0
4-Mild decline,31692.0,2803.778619,6900.005636,30.0,211.0,753.0,2595.00,244931.0
5-Sharp decline,36204.0,2613.579825,7929.019942,30.0,123.0,437.0,2117.00,617124.0


In [14]:
"""
gsc_avg_position averages across a page's queries,
but since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers),
a page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries.
This likely explains some of the scrambled middle-bucket ordering — sum_position,
while confounded by query volume, doesn't suffer from this specific averaging distortion.
"""

"\ngsc_avg_position averages across a page's queries,\nbut since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers),\na page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries.\nThis likely explains some of the scrambled middle-bucket ordering — sum_position,\nwhile confounded by query volume, doesn't suffer from this specific averaging distortion.\n"

In [15]:
df_trend.columns

Index(['gsc_impressions_feb', 'client_hash_id', 'content_hash_id',
       'gsc_impressions_mar', 'trend_pct'],
      dtype='object')

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
position_share = df['gsc_sum_position'] / df['gsc_sum_position'].sum()

cap_value = position_share.quantile(0.99)
position_share_capped = position_share.clip(upper=cap_value)

score = -df['trend_pct'] * position_share_capped * 1000
df['score']=score
df['score'] = df['score'] * 1000

In [17]:
df = df[['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score']].copy()

In [18]:
conditions = [
    (df['trend_pct'] < negative_mean) & (position_share > position_share.median()),
    (df['trend_pct'] < negative_mean) & (position_share <= position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0) & (position_share > position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0),
    (df['trend_pct'] >= 0) & (position_share > position_share.median()),
]
codes = [
    'strong declining trend with low page position',
    'strong declining trend',
    'mild declining trend with low page position',
    'mild declining trend',
    'low page position',
]
df['reason_code'] = np.select(conditions, codes, default='STABLE')


NameError: name 'negative_mean' is not defined

In [ ]:
df['action'] = np.select(
    [
        df['reason_code'] == 'strong declining trend with low page position',
        df['reason_code'].isin(['strong declining trend', 'mild declining trend with low page position']),
    ],
    ['REFRESH', 'MONITOR'],
    default='SKIP'
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
import os
os.makedirs('work/outputs', exist_ok=True)

output_cols = ['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score', 'reason_code', 'action']
ranked_queue = df.sort_values(by='score', ascending=False)[output_cols]
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

In [ ]:
ranked_queue.head(20)

## 3. Top-20 review

For each of the top 20, rows fall into three categories based on what's actually driving the score:

- **Clip-floor** (trend_pct near -99% to -100%): main risk is an unstable/unrepresentative February baseline making the drop look more extreme than it is
- **Large position** (sum_position notably high, near or contributing heavily to rank): main risk is query-volume inflating the sum rather than genuine ranking severity
- **Mid-range** (both signals moderate and proportionate): the most defensible flags, hardest to argue are false positives

| # | content_hash_id | trend_pct | sum_position | score | Category | Confidence | What would make it wrong |
|---|---|---|---|---|---|---|---|
| 1 | content_39e19a3ec2d95f9d | -99.99% | 379,794 | 5473.22 | Clip-floor | High | Feb baseline may have been a temporary spike; a near-total drop off an unstable base can look catastrophic without being genuine decay |
| 2 | content_c27cc4cb0d258665 | -99.96% | 260,994 | 5471.57 | Clip-floor | High | Same Feb-baseline risk; worth checking if Feb impressions sat near the ≥30 guard threshold, where % swings get noisy |
| 3 | content_d0633f4187569021 | -99.95% | 186,808 | 5471.07 | Clip-floor | High | Same as above — near-total drop, mid-pack position, main risk is an unrepresentative Feb baseline |
| 4 | content_f0703fc6ae385591 | -99.91% | 550,324 | 5469.02 | Large position | High | This page's sum_position is close to the 99th-percentile cap — its true value may be even higher, artificially flattened by the cap |
| 5 | content_660fe2b474b35a4c | -99.77% | 179,542 | 5461.06 | Clip-floor | High | Same Feb-baseline caveat as rows 1-3 |
| 6 | content_b49acf92cc1c8c7e | -99.76% | 205,726 | 5460.61 | Clip-floor | High | Same Feb-baseline caveat |
| 7 | content_cd3d932d4e1c8db0 | -99.67% | 699,631 | 5455.79 | Large position | Moderate | Highest sum_position in the top 20 — likely reflects genuinely high query volume rather than decline severity alone; worth confirming query count isn't inflating this artificially |
| 8 | content_c67c7e38bda3567e | -99.55% | 236,045 | 5449.06 | Mid-range | High | Both signals moderate-to-strong and roughly proportionate — one of the more defensible flags in the list |
| 9 | content_74de5f247659e956 | -99.49% | 237,466 | 5445.90 | Mid-range | High | Same reasoning as row 8 — balanced, defensible |
| 10 | content_3bfb3f753bcda98d | -99.37% | 397,424 | 5439.30 | Mid-range | Moderate | Trend slightly less extreme than rows above; still solid but closer to the "typical" decline case than the clip-floor extremes |
| 11 | content_9540d884af3e41fd | -99.25% | 659,435 | 5433.00 | Large position | Moderate | High sum_position is doing real work in this rank — check whether this reflects broad query coverage or a data artifact |
| 12 | content_164c1f53f13bcee1 | -99.15% | 2,098,750 | 5427.27 | Large position | Low-moderate | By far the largest sum_position in this top 20 (2.1M, ~3-6x the others) — strong candidate for a query-volume artifact rather than a uniquely bad page; worth manually checking this one's query count |
| 13 | content_a8024a89870ef783 | -99.09% | 688,455 | 5424.00 | Large position | Moderate | Same reasoning as rows 7/11 |
| 14 | content_65b8a4998e633d89 | -98.99% | 684,970 | 5418.54 | Large position | Moderate | Same reasoning as rows 7/11/13 |
| 15 | content_2392ac0360e7c84c | -98.91% | 246,070 | 5414.31 | Mid-range | High | Balanced signals, defensible flag |
| 16 | content_599eba17030c875c | -98.86% | 177,818 | 5411.26 | Mid-range | High | Balanced signals, defensible flag |
| 17 | content_236461caa443bdc3 | -98.74% | 203,537 | 5404.70 | Mid-range | High | Balanced signals, defensible flag |
| 18 | content_5671c3476176cbfd | -98.56% | 231,851 | 5395.06 | Mid-range | High | Balanced signals, defensible flag |
| 19 | content_23a42776a7009b65 | -98.47% | 276,849 | 5390.29 | Mid-range | High | Balanced signals, defensible flag |
| 20 | content_37d8fac68a8797e3 | -98.36% | 735,862 | 5384.09 | Large position | Moderate | Sizable sum_position relative to trend severity — check query volume before treating as top-tier urgent |

**Summary:** 6 of 20 rows (4, 7, 11, 12, 13, 14, 20) lean on a large `sum_position` more than trend severity alone, with row 12 the most extreme (2.1M — several times larger than its neighbors) and the best candidate for manual spot-checking. The remaining rows split between clip-floor cases (main risk: unstable February baseline) and well-balanced mid-range cases, which are the most defensible flags in the list.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# check row 12's suspected volume-artifact
df[df['content_hash_id'] == 'content_164c1f53f13bcee1']

# check a clip-floor row's actual Feb impressions (not just trend_pct)
df_trend[df_trend['content_hash_id'] == 'content_39e19a3ec2d95f9d'][['gsc_impressions_feb', 'gsc_impressions_mar', 'trend_pct']]

Initial hypothesis: rows sitting at the trend_pct clip floor (~-99% to -100%) might be weak picks, since a small/noisy February baseline can make a percentage drop look more dramatic than it really is.

Checked directly against Row 1 (`content_39e19a3ec2d95f9d`, top-ranked pick): Feb impressions = 42,185, Mar impressions = 5, trend_pct = -99.99%. This is a large, reliable baseline with a genuine, dramatic collapse — not a small-denominator artifact. This row is not a weak pick; it's one of the most defensible flags in the queue. The "unstable baseline" risk should be checked per-row, not assumed for the whole clip-floor category.

The stronger weak-pick candidate remains **Row 12** (`content_164c1f53f13bcee1`), whose `gsc_sum_position` (2,098,750) is 3-6x larger than its neighbors in the top 20. This is more likely a query-volume artifact (a page tracked across an unusually large number of queries) inflating its position score, rather than genuinely worse ranking performance. This row should be manually reviewed before acting on it as top priority.

**Leakage check**

This rule uses only two signals:
- `trend_pct` — computed from February vs. March `gsc_impressions`, both historical relative to the working month (March 2026), guarded to `feb_impressions >= 30`, clipped at the 99th percentile
- `gsc_sum_position` — March monthly total, aggregated from daily rows

Neither depends on the sealed June test window (`fact_content_query_90d`, permanently excluded), the `needs_refresh` label definition, or any post-March snapshot fields. Verified directly that the Week 3-excluded leakage-risk columns (`content_updated_date`, `last_optimized_date`, `optimization_eligible_date`) are not present in the working dataframe used for scoring:

```python
excluded = ['content_updated_date', 'last_optimized_date', 'optimization_eligible_date']
[col for col in excluded if col in df.columns]
# -> []
```

No FlyRank product-defined flags (e.g. the actual refresh/CTR-fix flag outputs) were used as inputs — only the underlying raw signals (impressions, position) that inform them, consistent with building an independent baseline rather than replicating an existing flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.